# Stage 4 Detection — GPT-5.5 (low reasoning) vs Molmo2

Head-to-head on the REAL Stage 4 metric: the frozen 20 Gupta test sheets, production-style
tiling, point-in-GT-box matching, stitch + dedup — the same harness that produced the existing
reference numbers (Molmo2 zero-shot F1 = 0.434, claude-sonnet-4-6 F1 = 0.380, both with the
v1 unengineered prompt; later runs use v3-style prompts).

**How to use:** set `MODEL` in the config cell and Run All; run the notebook once per model.
Each run appends one row to the shared results CSV on the HF dataset repo, so the comparison
accumulates. GPT-5.5 needs no GPU (CPU runtime fine — it's all API calls); Molmo2 needs a GPU.

**Comparability notes (honest):**
- Both models get the same v3-style scan-then-fenced-JSON prompt family and the same scoring.
  Molmo2 additionally supports its native `<points>` output — `MOLMO_OUTPUT = "points"` uses
  that (matches the recorded 0.434 methodology); boxes from GPT-5.5 are scored by their center
  point against GT boxes, the same unified point-in-box rule used all along.
- `TILE_SIZE = 1024` is the production-comparable default. `512 + upscale + enhance` is the
  known ~2.3x Molmo2 improvement config — run it as a second row, don't mix into the baseline.
- GPT-5.5 cost: 20 sheets ≈ 127 tiles ≈ 127 vision calls per run.

## 1. Config

In [ ]:
MODEL = "molmo2"          # "gpt-5.5" | "molmo2"

HF_TOKEN = "paste-your-hf-token-here"           # results CSV push
OPENAI_API_KEY = "paste-your-openai-key-here"   # only for MODEL="gpt-5.5"

GPT_REASONING_EFFORT = "low"    # matches reported production config
MOLMO_OUTPUT = "points"          # "points" (native, matches recorded 0.434) | "boxes"

# Tiling config. Production-comparable: 1024/205/1/False.
# Molmo2 dense-fix config: 512/102/2/True (report as a separate row).
TILE_SIZE = 512
OVERLAP = 102
UPSCALE = 2
ENHANCE = True

# Run only a subset of the 20 test sheets (e.g. skip dense sheets already benchmarked
# in another config). Empty list = run all 20. The CSV row records the exclusion.
EXCLUDE_SHEETS = ["151", "216", "233"]

DATA_REPO = "timthy45/pnid-extraction-datasets"
RESULTS_PATH_IN_REPO = "benchmarks/stage4_detection_results.csv"

assert HF_TOKEN.startswith("hf_"), "paste HF token"
if MODEL == "gpt-5.5":
    assert OPENAI_API_KEY.startswith("sk-"), "paste OpenAI key"

In [ ]:
!pip install -q transformers==4.57.1 accelerate

## 2. Fetch the frozen 20 test sheets (GitHub fixtures — no Drive, no big download)

In [ ]:
!pip install -q openai huggingface_hub
# Molmo2's remote-code processor requires this exact transformers version (newer
# versions reject its config's image_use_col_tokens key). Harmless for the GPT path.
# If transformers was already imported before this install: Runtime -> Restart, rerun.
!pip install -q transformers==4.57.1 accelerate

from pathlib import Path

SHEETS = Path("/content/gupta_test/sheets"); SHEETS.mkdir(parents=True, exist_ok=True)
LABELS = Path("/content/gupta_test/labels"); LABELS.mkdir(parents=True, exist_ok=True)

BASE = "https://raw.githubusercontent.com/TomGeorge45/pid-opensrc-substitution/main/notebooks/stage4/gupta_test_sheets"
TEST_IDS = ["0","103","11","124","129","136","145","148","15","151",
            "157","158","159","176","188","192","194","196","216","233"]

for sid in TEST_IDS:
    !wget -q -O {SHEETS}/{sid}.jpg {BASE}/sheets/{sid}.jpg
    !wget -q -O {LABELS}/{sid}.txt {BASE}/labels/{sid}.txt

for sid in TEST_IDS:
    assert (SHEETS/f"{sid}.jpg").stat().st_size > 1000, f"{sid}.jpg missing/empty"
    assert (LABELS/f"{sid}.txt").stat().st_size > 0, f"{sid}.txt missing/empty"
print("All 20 sheets + labels fetched.")

## 3. Tiling / scoring / stitch harness (same rules as every prior run)

In [ ]:
import json, re, time, csv, datetime
from PIL import Image, ImageOps

Image.MAX_IMAGE_PIXELS = None
DEDUP_DISTANCE_PX = 30
MIN_TILE_SIDE = 64

def compute_tile_grid(img_w, img_h, tile_size=TILE_SIZE, overlap=OVERLAP):
    stride = tile_size - overlap
    tiles = []
    y0 = 0
    while y0 < img_h:
        y1 = min(y0 + tile_size, img_h)
        x0 = 0
        while x0 < img_w:
            x1 = min(x0 + tile_size, img_w)
            if (x1 - x0) >= MIN_TILE_SIDE and (y1 - y0) >= MIN_TILE_SIDE:
                tiles.append((x0, y0, x1, y1))
            x0 += stride
        y0 += stride
    return tiles

def load_gt_boxes(label_path, img_w, img_h):
    boxes = []
    for line in Path(label_path).read_text().splitlines():
        if not line.strip():
            continue
        parts = line.split()
        cx, cy, w, h = (float(v) for v in parts[1:5])
        cx, cy, w, h = cx*img_w, cy*img_h, w*img_w, h*img_h
        boxes.append([cx-w/2, cy-h/2, cx+w/2, cy+h/2])
    return boxes

def stitch_and_dedup(preds):
    """preds: list of {point:(x,y), confidence} in SHEET coords. Proximity clustering."""
    kept = []
    for p in sorted(preds, key=lambda d: -d["confidence"]):
        x, y = p["point"]
        if all((x-k["point"][0])**2 + (y-k["point"][1])**2 > DEDUP_DISTANCE_PX**2 for k in kept):
            kept.append(p)
    return kept

def score_sheet(preds, gt_boxes):
    """Greedy one-to-one point-in-GT-box matching, confidence-ranked."""
    matched_gt = set()
    matched = 0
    for p in sorted(preds, key=lambda d: -d["confidence"]):
        x, y = p["point"]
        for gi, b in enumerate(gt_boxes):
            if gi in matched_gt:
                continue
            if b[0] <= x <= b[2] and b[1] <= y <= b[3]:
                matched_gt.add(gi)
                matched += 1
                break
    return matched

def enhance_tile(img):
    return ImageOps.autocontrast(img.convert("L")).convert("RGB")

## 4. Model adapter — GPT-5.5 (API) or Molmo2 (local GPU)

In [ ]:
import base64, io

BOX_PROMPT_TEMPLATE = """You are an expert P&ID reviewer analyzing one tile cropped from a larger Piping & Instrumentation Diagram. The image is {W}x{H} pixels.

First, briefly scan the tile region by region (top-left, top-right, center, bottom-left, bottom-right) and note what equipment symbols you see in each region. Dense tiles can contain 50+ symbols - keep going until every region is covered.

Count as a symbol: valves, instrument circles/bubbles, flanges, nozzles, safety devices, reducers, and other discrete equipment icons.
Not symbols: plain pipe/line segments, text labels alone, dimension arrows, borders.

Then output your final answer as a JSON array of arrays inside a ```json code fence. Each inner array is exactly: [x0, y0, x1, y1, confidence, "entity_type"] - tight boxes around just the icon, absolute pixel coordinates in the {W}x{H} image (top-left origin), confidence 0.0-1.0. If no symbols: []"""

POINTS_PROMPT = ("Point to every distinct P&ID equipment symbol in this tile: valves, instrument "
                 "circles/bubbles, flanges, nozzles, safety devices, reducers, and other discrete "
                 "equipment icons. Do NOT point at plain pipe segments, text labels, dimension "
                 "arrows, or borders. Dense tiles can contain 50+ symbols - find them all.")

def parse_fenced_boxes(text):
    """-> (list of {point,(cx,cy) from box center, confidence}, err)"""
    fenced = re.findall(r'```(?:json)?\s*(\[[\s\S]*?\])\s*```', text)
    cleaned = fenced[-1].strip() if fenced else re.sub(r'^```(?:json)?\s*|\s*```$', '', text.strip())
    try:
        data = json.loads(cleaned)
    except json.JSONDecodeError as e:
        return None, f"JSONDecodeError: {e}"
    if not isinstance(data, list):
        return None, "not a list"
    out = []
    for item in data:
        if not (isinstance(item, list) and len(item) >= 4):
            continue
        x0, y0, x1, y1 = (float(v) for v in item[:4])
        conf = float(item[4]) if len(item) > 4 and isinstance(item[4], (int, float)) else 0.5
        out.append({"point": ((x0+x1)/2, (y0+y1)/2), "confidence": conf})
    return out, None

_MOLMO_POINTS_RE = re.compile(r'<(?:points|tracks).*? coords="([0-9\t:;, .]+)"/?>')

def parse_molmo_points(text, w, h):
    """Molmo2 real format: <points coords="frame x y; frame x y; ..."/>, values scaled by 1000.
    Ported from src/stage4_symbol_detection/molmo_candidate.py (the parser behind the recorded
    0.434 run), including the duplicated-leading-index repair."""
    out = []
    for m in _MOLMO_POINTS_RE.finditer(text):
        nums = [float(v) for v in re.split(r'[\t:;, ]+', m.group(1).strip()) if v]
        if len(nums) % 3 != 0:
            if len(nums) >= 2 and nums[0] == nums[1] and (len(nums) - 1) % 3 == 0:
                nums = nums[1:]
            else:
                return None, f"coords not a multiple of 3: {nums[:9]}..."
        for i in range(0, len(nums), 3):
            _frame, xs, ys = nums[i:i+3]
            out.append({"point": (xs/1000*w, ys/1000*h), "confidence": 0.5})
    if not out and "<point" in text.lower():
        return None, "point-like tags present but regex matched nothing"
    return out, None


## 5. Run all 20 sheets — tiled, remapped, stitched, scored

In [ ]:
total_gt = total_pred = total_matched = 0
parse_failures = total_tiles = 0
per_sheet = []
tile_records = []   # raw per-tile predictions, persisted to HF for tile-level comparison
run_t0 = time.time()

RUN_IDS = [s for s in TEST_IDS if s not in EXCLUDE_SHEETS]
print(f'running {len(RUN_IDS)}/{len(TEST_IDS)} sheets (excluded: {EXCLUDE_SHEETS or "none"})')
for sid in RUN_IDS:
    img = Image.open(SHEETS/f"{sid}.jpg").convert("RGB")
    W, H = img.size
    gt = load_gt_boxes(LABELS/f"{sid}.txt", W, H)

    sheet_preds = []
    tiles = compute_tile_grid(W, H)
    for (x0, y0, x1, y1) in tiles:
        total_tiles += 1
        tile = img.crop((x0, y0, x1, y1))
        if UPSCALE > 1:
            tile = tile.resize((tile.width*UPSCALE, tile.height*UPSCALE), Image.LANCZOS)
        if ENHANCE:
            tile = enhance_tile(tile)
        try:
            preds, err = predict_tile(tile)
        except Exception as e:
            preds, err = None, f"{type(e).__name__}: {e}"
        if preds is None:
            parse_failures += 1
            print(f"  [parse-fail] sheet {sid} tile ({x0},{y0}): {str(err)[:80]}")
            continue
        scale = 1.0 / UPSCALE
        remapped = [{"point": (p["point"][0]*scale + x0, p["point"][1]*scale + y0),
                     "confidence": p["confidence"]} for p in preds]
        sheet_preds.extend(remapped)
        tile_records.append({"sheet": sid, "tile": [x0, y0, x1, y1],
                             "n_preds": len(remapped),
                             "preds_sheet_coords": [[round(p["point"][0],1), round(p["point"][1],1),
                                                     p["confidence"]] for p in remapped]})

    kept = stitch_and_dedup(sheet_preds)
    matched = score_sheet(kept, gt)
    total_gt += len(gt); total_pred += len(kept); total_matched += matched
    P = matched/len(kept) if kept else 0.0
    R = matched/len(gt) if gt else 0.0
    F = 2*P*R/(P+R) if P+R else 0.0
    per_sheet.append((sid, len(tiles), len(gt), len(kept), matched, P, R, F))
    print(f"{sid:>4s} tiles={len(tiles):2d} gt={len(gt):4d} pred={len(kept):4d} "
          f"matched={matched:4d} P={P:.2f} R={R:.2f} F1={F:.2f}")

P = total_matched/total_pred if total_pred else 0.0
R = total_matched/total_gt if total_gt else 0.0
F1 = 2*P*R/(P+R) if P+R else 0.0
print(f"\n=== {MODEL} | tile={TILE_SIZE} upscale={UPSCALE} enhance={ENHANCE} ===")
print(f"micro P={P:.3f} R={R:.3f} F1={F1:.3f}  parse_failures={parse_failures}/{total_tiles} "
      f"({100*parse_failures/max(total_tiles,1):.1f}%)  elapsed={time.time()-run_t0:.0f}s")
print("\nReference rows (v1 unengineered prompt, 1024 tiles): Molmo2 F1=0.434, claude-sonnet-4-6 F1=0.380. Pass bar: F1>=0.70.")

## 6. Append to shared results CSV on HF

In [ ]:
from huggingface_hub import HfApi, hf_hub_download

RESULTS_LOCAL = Path("/content/stage4_detection_results.csv")
FIELDS = ["timestamp","model","prompt","tile_size","upscale","enhance",
          "precision","recall","f1","parse_failures","tiles","test_set","notes"]
try:
    prev = hf_hub_download(repo_id=DATA_REPO, filename=RESULTS_PATH_IN_REPO,
                           repo_type="dataset", token=HF_TOKEN)
    RESULTS_LOCAL.write_text(Path(prev).read_text())
except Exception:
    with open(RESULTS_LOCAL, "w", newline="") as f:
        csv.DictWriter(f, fieldnames=FIELDS).writeheader()

with open(RESULTS_LOCAL, "a", newline="") as f:
    csv.DictWriter(f, fieldnames=FIELDS).writerow({
        "timestamp": datetime.datetime.utcnow().isoformat(timespec="seconds"),
        "model": MODEL + (f"-{GPT_REASONING_EFFORT}" if MODEL == "gpt-5.5" else f"-{MOLMO_OUTPUT}"),
        "prompt": "v3-scan-fenced" if MODEL == "gpt-5.5" or MOLMO_OUTPUT == "boxes" else "points-v3",
        "tile_size": TILE_SIZE, "upscale": UPSCALE, "enhance": ENHANCE,
        "precision": round(P,4), "recall": round(R,4), "f1": round(F1,4),
        "parse_failures": f"{parse_failures}/{total_tiles}",
        "tiles": total_tiles, "test_set": "test_ids.json v1 (frozen 20)",
        "notes": (f"excluded:{','.join(EXCLUDE_SHEETS)}" if EXCLUDE_SHEETS else "")})

api = HfApi(token=HF_TOKEN)
api.upload_file(path_or_fileobj=str(RESULTS_LOCAL), path_in_repo=RESULTS_PATH_IN_REPO,
                repo_id=DATA_REPO, repo_type="dataset")

# persist raw per-tile predictions so future runs can be compared tile-for-tile
run_tag = f"{MODEL}_tile{TILE_SIZE}_up{UPSCALE}_enh{ENHANCE}_{datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%S')}"
PREDS_LOCAL = Path(f"/content/preds_{run_tag}.json")
PREDS_LOCAL.write_text(json.dumps({"model": MODEL, "tile_size": TILE_SIZE, "upscale": UPSCALE,
                                   "enhance": ENHANCE, "tiles": tile_records}, indent=1))
api.upload_file(path_or_fileobj=str(PREDS_LOCAL),
                path_in_repo=f"benchmarks/stage4_tile_predictions/{run_tag}.json",
                repo_id=DATA_REPO, repo_type="dataset")
print(f"raw per-tile predictions saved: benchmarks/stage4_tile_predictions/{run_tag}.json")
print("results pushed. All rows so far:")
import csv as _csv
for row in _csv.DictReader(open(RESULTS_LOCAL)):
    print(f"  {row['model']:22s} tile={row['tile_size']:>4s} up={row['upscale']} enh={row['enhance']:5s} "
          f"P={row['precision']} R={row['recall']} F1={row['f1']} pf={row['parse_failures']}")